# Notebook 03_2 — Predictive Performance (Text-Only Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Women's Shoes (Size 8)

---

This notebook evaluates how well different model specifications predict:
- **Q_t** — Sales rank (demand proxy)
- **P_bb_t** — Buybox price

Model specifications compared (following paper Table 2):
- OLS / Boosting with Tabular features only
- OLS / Boosting with Tabular + PCA (5 components)
- OLS / Boosting with Tabular + Cluster Similarities
- Deep Time Independent (RoBERTa text embeddings)
- Deep Time Dependent (RoBERTa text + lag1 embeddings)

R² evaluated on both level (Q_t, P_bb_t) and first-difference (ΔQ_t, ΔP_bb_t) models.

## ① Mount Drive

In [1]:
# Local mode - no Google Drive needed
print('Local mode')

Local mode


## ② Set Working Directory

In [2]:
import os, sys
from pathlib import Path

# Local mode - no Google Drive needed
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root")

CODE_DIR = str(PROJECT_ROOT / 'code')
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print(f'Working directory: {os.getcwd()}')

Working directory: /home/iankuzuma/claude_code/demand_modeling/women-8-subcat-split-proper-embedding/fashion-sneakers/code


## ③ Imports

In [3]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)
print('✅ Imports done')

✅ Imports done


## ④ Load Dataset

Loads the text-only embeddings dataset from notebook 01_2.
`dropna()` removes first-period rows with no lag features.

In [4]:
txt_only = True
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"

df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val   = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val   = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val   = df_full_val.dropna()

dummy_subcat_names = [category for category in df_full_val["subcat_aggregated"].unique()]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Train shape: {df_full_train.shape}")
print(f"Val shape:   {df_full_val.shape}")
print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Train shape: (5160, 323)
Val shape:   (5160, 323)
Dummy subcat names: ['Fashion Sneakers']
All time steps: ['2025-04-28', '2025-05-26', '2025-06-23', '2025-07-21', '2025-08-18', '2025-09-15', '2025-10-13', '2025-11-10', '2025-12-08', '2026-01-05', '2026-02-02', '2026-03-02']


Index(['ASIN', 'date', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=323)

## ⑤ Define Controls and Feature Specifications

Controls follow paper Section 3:
- Continuous: RATING, REVIEW_COUNT, offer counts
- Time dummies (excluding first period as baseline)
- Subcat dummies (excluding Residual as baseline)
- PCA components (5) and cluster similarities (5) as additional features

In [5]:
n_lags = 1

outcome   = "Q_t"
treatment = "P_bb_t"

outcome_diff   = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = ["RATING", "REVIEW_COUNT"]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0", "similarity_cluster_1", "similarity_cluster_2",
    "similarity_cluster_3", "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls
additional_controls_deep = [var for var in additional_controls if var not in dummy_subcat_names]
controls_emb = [var for var in df_full_train.columns if "emb" in var]

print(f"Continuous controls:    {len(cont_controls)}")
print(f"Dummy controls:         {len(dummy_controls)}")
print(f"Total controls:         {len(additional_controls)}")
print(f"Embedding columns:      {len(controls_emb)}")

Continuous controls:    5
Dummy controls:         13
Total controls:         18
Embedding columns:      256


## ⑥ Initialize Results DataFrames

In [6]:
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df      = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)
print('✅ Results DataFrames initialized')

✅ Results DataFrames initialized


## ⑦ Deep Model R² — Level

Evaluates how well the deep learning predictions (from Part 5) explain
sales rank and price in both time-independent and lag1 configurations.

In [7]:
df_dict = {"Train": df_full_train, "Test": df_full_val}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome (time independent):  {r2_ml_l}")
    print(f"  R2 Treatment (time independent): {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"  R2 Outcome (lag1):               {r2_ml_l_lag1}")
    print(f"  R2 Treatment (lag1):             {r2_ml_m_lag1}")
    print()

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
  R2 Outcome (time independent):  0.6367
  R2 Treatment (time independent): 0.5881
  R2 Outcome (lag1):               0.8153
  R2 Treatment (lag1):             0.7223

Evaluation for Test set
  R2 Outcome (time independent):  0.4148
  R2 Treatment (time independent): 0.4946
  R2 Outcome (lag1):               0.8371
  R2 Treatment (lag1):             0.66



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.6367,0.4148,0.5881,0.4946
Deep Time Dependent,0.8153,0.8371,0.7223,0.6600


## ⑧ Deep Model R² — Diff

Same evaluation on first-difference outcomes (ΔQ_t, ΔP_bb_t).

In [8]:
results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome diff (time independent):  {r2_ml_l_diff}")
    print(f"  R2 Treatment diff (time independent): {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"  R2 Outcome diff (lag1):               {r2_ml_l_diff_lag1}")
    print(f"  R2 Treatment diff (lag1):             {r2_ml_m_diff_lag1}")
    print()

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
  R2 Outcome diff (time independent):  0.001563
  R2 Treatment diff (time independent): -0.00486003
  R2 Outcome diff (lag1):               0.06264867
  R2 Treatment diff (lag1):             -0.003287

Evaluation for Test set
  R2 Outcome diff (time independent):  0.0009672
  R2 Treatment diff (time independent): -0.00660125
  R2 Outcome diff (lag1):               0.05950223
  R2 Treatment diff (lag1):             -0.00529559



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.001563,0.000967,-0.004860,-0.006601
Deep Time Dependent,0.062649,0.059502,-0.003287,-0.005296


## ⑨ Build Feature Matrices

Constructs train and test feature matrices for tabular, PCA, and similarity
specifications used in OLS and Boosting models.

In [9]:
# Train set
y_train       = df_full_train[outcome].squeeze()
d_train       = df_full_train[treatment].squeeze()
y_train_diff  = df_full_train[outcome_diff].squeeze()
d_train_diff  = df_full_train[treatment_diff].squeeze()

x_train     = sm.add_constant(df_full_train[additional_controls])
x_train_pca = sm.add_constant(df_full_train[additional_controls + controls_pca])
x_train_sim = sm.add_constant(df_full_train[additional_controls + controls_similarities])

# Test set
y_test      = df_full_val[outcome].squeeze()
d_test      = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test     = sm.add_constant(df_full_val[additional_controls])
x_test_pca = sm.add_constant(df_full_val[additional_controls + controls_pca])
x_test_sim = sm.add_constant(df_full_val[additional_controls + controls_similarities])

# Rename columns for LightGBM (no special characters)
for df in [x_train, x_test, x_train_pca, x_test_pca, x_train_sim, x_test_sim]:
    df.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x), inplace=True)

print(f"x_train shape:     {x_train.shape}")
print(f"x_train_pca shape: {x_train_pca.shape}")
print(f"x_train_sim shape: {x_train_sim.shape}")

x_train shape:     (5160, 18)
x_train_pca shape: (5160, 23)
x_train_sim shape: (5160, 23)


## ⑩ Build Dict Structures

In [10]:
train_dict = {
    "y": y_train, "y_diff": y_train_diff,
    "d": d_train, "d_diff": d_train_diff,
    "x": x_train, "x_pca": x_train_pca, "x_sim": x_train_sim,
}

test_dict = {
    "y": y_test, "y_diff": y_test_diff,
    "d": d_test, "d_diff": d_test_diff,
    "x": x_test, "x_pca": x_test_pca, "x_sim": x_test_sim,
}
print('✅ Train and test dicts ready')

✅ Train and test dicts ready


## ⑪ Tabular Models — Level

Compares OLS and LightGBM Boosting across three feature specifications:
- **Tabular** — continuous + time/subcat dummies only
- **Tabular + PCA** — adds 5 PCA components from embeddings
- **Tabular + Similarities** — adds 5 cluster similarity scores

In [11]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y"])
    print(f"  R2 Outcome train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d"])
    print(f"  R2 Treatment train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_tab = pd.concat([results_df_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
]
results_df_tab.columns = column_names
results_df_tab.index   = row_names
results_df_tab


 Feature specification: x
  OLS
  R2 Outcome train/test: 0.1632 / 0.0079
  R2 Treatment train/test: 0.2053 / 0.1122
  Boosting


  R2 Outcome train/test: 0.7559 / 0.6152


  R2 Treatment train/test: 0.6794 / 0.4724

 Feature specification: x_pca
  OLS
  R2 Outcome train/test: 0.6193 / 0.4821
  R2 Treatment train/test: 0.5276 / 0.4652
  Boosting


  R2 Outcome train/test: 0.9336 / 0.6803


  R2 Treatment train/test: 0.9526 / 0.5551

 Feature specification: x_sim
  OLS
  R2 Outcome train/test: 0.6192 / 0.4781
  R2 Treatment train/test: 0.5193 / 0.4637
  Boosting


  R2 Outcome train/test: 0.9261 / 0.6548


  R2 Treatment train/test: 0.9408 / 0.5103


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.163191,0.007858,0.205252,0.112189
Boosting (Tabular),0.755899,0.615160,0.679386,0.472431
OLS (Tabular + PCA),0.619316,0.482069,0.527626,0.465207
Boosting (Tabular + PCA),0.933612,0.680311,0.952640,0.555071
OLS (Tabular + Similarities),0.619232,0.478103,0.519342,0.463713
Boosting (Tabular + Similarities),0.926093,0.654846,0.940815,0.510273


## ⑫ Summary — Level Models

In [12]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.163191,0.007858,0.205252,0.112189
Boosting (Tabular),0.755899,0.615160,0.679386,0.472431
OLS (Tabular + PCA),0.619316,0.482069,0.527626,0.465207
Boosting (Tabular + PCA),0.933612,0.680311,0.952640,0.555071
OLS (Tabular + Similarities),0.619232,0.478103,0.519342,0.463713
Boosting (Tabular + Similarities),0.926093,0.654846,0.940815,0.510273
Deep Time Independent,0.636700,0.414800,0.588100,0.494600
Deep Time Dependent,0.815300,0.837100,0.722300,0.660000


## ⑬ Tabular Models — Diff

Same comparison on first-difference outcomes (ΔQ_t, ΔP_bb_t).
First differences remove product fixed effects and time trends.

In [13]:
feature_specifications = ["x", "x_pca", "x_sim"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y_diff"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome diff train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d_diff"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d_diff"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment diff train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y_diff"])
    print(f"  R2 Outcome diff train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d_diff"])
    print(f"  R2 Treatment diff train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_diff_tab = pd.concat([results_df_diff_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
]
results_df_diff_tab.columns = column_names
results_df_diff_tab.index   = row_names
results_df_diff_tab


 Feature specification: x
  OLS
  R2 Outcome diff train/test: 0.1234 / 0.1091
  R2 Treatment diff train/test: 0.0113 / 0.0043
  Boosting
  R2 Outcome diff train/test: 0.5665 / 0.3007


  R2 Treatment diff train/test: 0.2749 / 0.0008

 Feature specification: x_pca
  OLS
  R2 Outcome diff train/test: 0.1253 / 0.1095
  R2 Treatment diff train/test: 0.0118 / 0.0043
  Boosting


  R2 Outcome diff train/test: 0.6897 / 0.3108


  R2 Treatment diff train/test: 0.3727 / -0.0050

 Feature specification: x_sim
  OLS
  R2 Outcome diff train/test: 0.1253 / 0.1092
  R2 Treatment diff train/test: 0.0128 / 0.0038
  Boosting


  R2 Outcome diff train/test: 0.6638 / 0.2727


  R2 Treatment diff train/test: 0.3539 / -0.0267


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.123431,0.109087,0.011345,0.004305
Boosting (Tabular),0.566545,0.300741,0.274928,0.000840
OLS (Tabular + PCA),0.125292,0.109514,0.011811,0.004297
Boosting (Tabular + PCA),0.689655,0.310843,0.372699,-0.004950
OLS (Tabular + Similarities),0.125261,0.109190,0.012758,0.003799
Boosting (Tabular + Similarities),0.663850,0.272740,0.353945,-0.026651


## ⑭ Summary — Diff Models

In [14]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.123431,0.109087,0.011345,0.004305
Boosting (Tabular),0.566545,0.300741,0.274928,0.000840
OLS (Tabular + PCA),0.125292,0.109514,0.011811,0.004297
Boosting (Tabular + PCA),0.689655,0.310843,0.372699,-0.004950
OLS (Tabular + Similarities),0.125261,0.109190,0.012758,0.003799
Boosting (Tabular + Similarities),0.663850,0.272740,0.353945,-0.026651
Deep Time Independent,0.001563,0.000967,-0.004860,-0.006601
Deep Time Dependent,0.062649,0.059502,-0.003287,-0.005296


## ⑮ Final Summary (% format)

R² values multiplied by 100 for readability — matches paper Table 2 format.

In [15]:
print("=== Level Models — Test R² (%) ===")
print(results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100)
print()
print("=== Diff Models — Test R² (%) ===")
print(results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100)

=== Level Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                           0.79      11.22
Boosting (Tabular)                     61.52      47.24
OLS (Tabular + PCA)                    48.21      46.52
Boosting (Tabular + PCA)               68.03      55.51
OLS (Tabular + Similarities)           47.81      46.37
Boosting (Tabular + Similarities)      65.48      51.03
Deep Time Independent                  41.48      49.46
Deep Time Dependent                    83.71      66.00

=== Diff Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          10.91       0.43
Boosting (Tabular)                     30.07       0.08
OLS (Tabular + PCA)                    10.95       0.43
Boosting (Tabular + PCA)               31.08      -0.50
OLS (Tabular + Similarities)           10.92       0.38
Boosting (Tabular + Similarities)      27.27      -2.67
Deep Time Independent             